# Compare iSSF (linear/MLP) vs deepSSF

Minimal notebook to:
- load checkpoints per study for `issf_linear`, `issf_mlp`, `deepssf`,
- run inference on **all test steps**,
- sample from each model’s probability surface:
  - deepSSF: 500 samples from the 101×101 grid,
  - iSSF: 10,000 candidates → 1,000 samples from candidate softmax.

Outputs (under `<output_dir>/compare_outputs/`):
- `summary.csv`
- `surface_samples.csv`


In [1]:
from __future__ import annotations

import hashlib, math, sys
from pathlib import Path

import numpy as np, pandas as pd, torch

BENCHMARK_CONFIG_PATH = "../configs/benchmarks/all.yaml"
ONLY_ONE_STUDY = True
STUDY_ID = "tsavo_lions"
SETUP_IF_MISSING = False
CHECKPOINT_ROOT_OVERRIDE: str | None = None

ISSF_SURFACE_NUM_CANDIDATES = 10_000
ISSF_SURFACE_NUM_SAMPLES = 1_000
DEEPSSF_SURFACE_NUM_SAMPLES = 500
SURFACE_STEPS_PER_STUDY: int | None = 10
SURFACE_SEED = 0
TOPK = (1, 5, 10, 25, 50)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "movebench").is_dir():
            return q
    raise FileNotFoundError("Could not find repo root")


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def rpath(p: str | Path) -> Path:
    p = Path(p)
    if p.exists():
        return p.resolve()
    for base in (REPO_ROOT, REPO_ROOT / "notebooks"):
        q = (base / p).resolve()
        if q.exists():
            return q
    raise FileNotFoundError(p)


def seed(*parts: object) -> int:
    s = "|".join(str(p) for p in parts).encode("utf-8")
    return int(hashlib.sha256(s).hexdigest()[:8], 16)


print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)


REPO_ROOT: /data/vision/beery/scratch/justin/movebench
DEVICE: cuda


In [2]:
from __future__ import annotations

from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Subset, Dataset

from movebench import Study
from movebench import benchmark as mb_benchmark
from movebench.datasets.deepssf import DeepSSFDataset
from movebench.features.normalization import load_points_normalization_mean_std
from movebench.features.points import select_point_store_dir
from movebench.geometry import centered_grid_coordinates_m
from movebench.models.deepssf import DeepSSF
from movebench.models.ssf import LinearStepSelect, MLPStepSelect
from movebench.train import collate_deepssf, collate_moveformer, collate_step_select, eval_deepssf, eval_step_select
from movebench.trajectories import SET_COL, TIME_COL, compute_movement_features, select_step_starts

import movebench.features.points as mb_points


cfg_path = rpath(BENCHMARK_CONFIG_PATH)
cfg = mb_benchmark.load_config(cfg_path)
methods = {str(m["kind"]): dict(m) for m in cfg["methods"]}
for k in ("issf_linear", "issf_mlp", "deepssf"):
    if k not in methods:
        raise ValueError(f"Benchmark config missing {k}")
if "moveformer" not in methods:
    print("NOTE: benchmark config missing moveformer; skipping")

layers_config = rpath(cfg["layers_config"])
studies = mb_benchmark.resolve_studies(cfg)
if ONLY_ONE_STUDY:
    sid = STUDY_ID or str(studies[0])
    studies = [sid]

out_dir = Path(str(cfg["output_dir"]))
out_dir = out_dir if out_dir.is_absolute() else (REPO_ROOT / out_dir).resolve()
out_cmp = out_dir / "compare_outputs"
out_cmp.mkdir(parents=True, exist_ok=True)
ckpt_root = Path(CHECKPOINT_ROOT_OVERRIDE) if CHECKPOINT_ROOT_OVERRIDE else (out_dir / "checkpoints")

print("cfg:", cfg_path)
print("layers_config:", layers_config)
print("output_dir:", out_dir)
print("checkpoint_root:", ckpt_root)
print("n_studies:", len(studies), studies)
if not ckpt_root.exists():
    print("\nWARNING: no checkpoints found; run:")
    print(f"  python {REPO_ROOT/'benchmark.py'} --config {cfg_path}")


def ckpt_path(study_id: str, kind: str) -> Path:
    return ckpt_root / study_id / f"{methods[kind]['name']}.pt"


def load_model(study_id: str, kind: str, *, cov_dim: int | None = None) -> tuple[torch.nn.Module, dict[str, Any]] | None:
    p = ckpt_path(study_id, kind)
    if not p.exists():
        return None
    d = torch.load(p, map_location="cpu")
    meta = dict(d.get("meta", {}) or {})
    state = dict(d.get("state_dict", {}) or {})
    if kind == "issf_linear":
        m = LinearStepSelect(int(meta["in_channels"]))
    elif kind == "issf_mlp":
        mc = dict(methods[kind].get("model", {}) or {})
        hd = tuple(int(x) for x in mc.get("hidden_dims", (128,)))
        dr = float(mc.get("dropout", 0.6))
        m = MLPStepSelect(int(meta["in_channels"]), hidden_dims=hd, dropout=dr)
    elif kind == "deepssf":
        m = DeepSSF(int(meta["env_channels"]), image_dim=int(meta["image_dim"]), pixel_size_m=float(meta["pixel_size_m"]), device=DEVICE)
    elif kind == "moveformer":
        from movebench.models.moveformer import build_moveformer_selection_model
        cd = int(cov_dim or meta.get("cov_dim", 0) or 0)
        if cd <= 0:
            raise ValueError("Need cov_dim to load moveformer")
        m = build_moveformer_selection_model(cov_dim=cd)
    else:
        raise ValueError(kind)
    m.load_state_dict(state)
    return m.to(DEVICE).eval(), meta


def get_sampler_hash(study: Study, plan_config_hash: str) -> str:
    s = cfg["datasets"]["issf"]["sampler"]
    try:
        _dir, m = select_point_store_dir(
            study_dir=study.study_dir,
            split="train",
            config_hash=str(plan_config_hash),
            expected_type="parametric",
            num_candidates=int(s["num_candidates"]),
            random_seed=int(s["random_seed"]),
            step_multiplier=float(s["step_multiplier"]),
            step_q=float(s["step_q"]),
            gap_q=float(s["gap_q"]),
            bounds_buffer_deg=float(s["bounds_buffer_deg"]),
        )
        return str(m.sampler_hash)
    except FileNotFoundError:
        if not SETUP_IF_MISSING:
            raise
        _ = mb_benchmark.ensure_study_ready(study=study, cfg=cfg)
        return get_sampler_hash(study, plan_config_hash)


def mv_stats_from_train_store(study: Study, plan_config_hash: str, sampler_hash: str) -> dict[str, float]:
    pt = study.study_dir / "features" / "points" / "train" / plan_config_hash / sampler_hash / "features.pt"
    d = torch.load(pt, map_location="cpu")
    sl = d["step_length_m"][:, -1].to(dtype=torch.float64)
    ta = d["turn_angle_rad"][:, -1].to(dtype=torch.float64)
    log_sl, cos_ta = torch.log(sl + 1e-8), torch.cos(ta)

    def ms(x: torch.Tensor) -> tuple[float, float]:
        m, s = float(x.mean().item()), float(x.std(unbiased=False).item())
        if not math.isfinite(s) or s < 1e-8:
            s = 1.0
        return m, s

    sl_m, sl_s = ms(sl)
    ls_m, ls_s = ms(log_sl)
    ct_m, ct_s = ms(cos_ta)
    return {"sl_mean": sl_m, "sl_std": sl_s, "log_sl_mean": ls_m, "log_sl_std": ls_s, "cos_ta_mean": ct_m, "cos_ta_std": ct_s}


def issf_step_x(env: torch.Tensor, valid: torch.Tensor, sl_m: torch.Tensor, ta: torch.Tensor, tsn: int, mv: dict[str, float], mean: torch.Tensor, std: torch.Tensor, issf_cfg: dict[str, Any]) -> torch.Tensor:
    if bool(issf_cfg.get("include_individual_id", False)):
        raise ValueError("Minimal notebook: include_individual_id not supported")
    P, Cenv = int(env.shape[1]), int(env.shape[0])
    if bool(issf_cfg.get("normalize", True)):
        env = torch.where(valid, (env - mean.view(-1, 1)) / std.view(-1, 1), torch.zeros_like(env))
    move = torch.stack([sl_m, torch.log(sl_m + 1e-8), torch.cos(ta)], dim=0)
    if bool(issf_cfg.get("normalize_movement", True)):
        move[0] = (move[0] - float(mv["sl_mean"])) / float(mv["sl_std"])
        move[1] = (move[1] - float(mv["log_sl_mean"])) / float(mv["log_sl_std"])
        move[2] = (move[2] - float(mv["cos_ta_mean"])) / float(mv["cos_ta_std"])
    parts, valids = [env, move], [valid, torch.ones((3, P), dtype=torch.bool)]
    time = None
    if bool(issf_cfg.get("include_time", False)):
        ts = pd.to_datetime(int(tsn), unit="ns", utc=True)
        doy, hf = int(ts.dayofyear), float(ts.hour) + float(ts.minute) / 60.0 + float(ts.second) / 3600.0
        time = torch.tensor([
            math.sin(2.0 * math.pi * doy / 365.25),
            math.cos(2.0 * math.pi * doy / 365.25),
            math.sin(2.0 * math.pi * hf / 24.0),
            math.cos(2.0 * math.pi * hf / 24.0),
        ], dtype=torch.float32).view(4, 1).expand(4, P)
        parts.append(time)
        valids.append(torch.ones((4, P), dtype=torch.bool))

    def inter(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        Ka, P = a.shape
        Kb = b.shape[0]
        return (a.unsqueeze(1) * b.unsqueeze(0)).reshape(Ka * Kb, P)

    parts.append(inter(move, env))
    valids.append(valid.unsqueeze(0).expand(3, -1, -1).reshape(3 * Cenv, P))
    if time is not None:
        parts.append(inter(time, env))
        valids.append(valid.unsqueeze(0).expand(4, -1, -1).reshape(4 * Cenv, P))
        parts.append(inter(time, move))
        valids.append(torch.ones((12, P), dtype=torch.bool))
    x, v = torch.cat(parts, dim=0), torch.cat(valids, dim=0)
    return torch.where(v, x, torch.zeros_like(x))


summary, surface = [], []
rng = np.random.default_rng(int(SURFACE_SEED))

for study_id in studies:
    print("Computing for", study_id)
    study = Study(study_id)
    plan = study.layer_plan(str(layers_config))
    sh = get_sampler_hash(study, str(plan.config_hash))
    mean, std = load_points_normalization_mean_std(study_dir=study.study_dir, plan=plan, sampler_hash=sh)
    mv = mv_stats_from_train_store(study, str(plan.config_hash), sh)

    models = {}
    metas = {}
    for k in ("issf_linear", "issf_mlp", "deepssf"):
        out = load_model(study_id, k)
        if out is not None:
            models[k], metas[k] = out
    has_mf_ckpt = ("moveformer" in methods) and ckpt_path(study_id, "moveformer").exists()
    if (not models) and (not has_mf_ckpt):
        continue

    issf_cfg = cfg["datasets"]["issf"]
    s = issf_cfg["sampler"]
    issf_test = study.issf_dataset(
        split="test",
        config=str(layers_config),
        normalize=bool(issf_cfg.get("normalize", True)),
        include_time=bool(issf_cfg.get("include_time", False)),
        include_individual_id=bool(issf_cfg.get("include_individual_id", False)),
        normalize_movement=bool(issf_cfg.get("normalize_movement", True)),
        num_candidates=int(s["num_candidates"]),
        random_seed=int(s["random_seed"]),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
        bounds_buffer_deg=float(s["bounds_buffer_deg"]),
    )
    
    # didn't ask for these
    # issf_loader = DataLoader(issf_test, batch_size=int(issf_cfg.get("batch_size", 32)), shuffle=False, 
    #                          num_workers=8, #  int(issf_cfg.get("num_workers", 0) or 0), 
    #                          collate_fn=collate_step_select)
    # if "issf_linear" in models:
    #     r = eval_step_select(models["issf_linear"], issf_loader, DEVICE, topk=TOPK)
    #     summary.append({"study_id": study_id, "kind": "issf_linear", **r})
    # if "issf_mlp" in models:
    #     r = eval_step_select(models["issf_mlp"], issf_loader, DEVICE, topk=TOPK)
    #     summary.append({"study_id": study_id, "kind": "issf_mlp", **r})

    dcfg = cfg["datasets"]["deepssf"]
    chips_test = study.chip_dataset(
        split="test",
        config=str(layers_config),
        mode="steps",
        output_hw=tuple(int(x) for x in dcfg.get("output_hw", (101, 101))),
        normalize=bool(dcfg.get("normalize", True)),
        normalization_sampler_hash=sh,
        step_multiplier=float(dcfg.get("step_multiplier", 2.0)),
        step_q=float(dcfg.get("step_q", 0.95)),
        gap_q=float(dcfg.get("gap_q", 0.999)),
    )
    deepssf_test = DeepSSFDataset(chips_test)

    mf_test = None
    if ("moveformer" in methods) and ckpt_path(study_id, "moveformer").exists():
        mfcfg = dict((cfg.get("datasets", {}) or {}).get("moveformer", {}) or {})
        mf_test = study.moveformer_dataset(
            split="test",
            config=str(layers_config),
            max_len=int(mfcfg.get("max_len", 500)),
            min_len=int(mfcfg.get("min_len", 2)),
            normalize_env=bool(mfcfg.get("normalize_env", True)),
            require_normalization_stats=bool(mfcfg.get("require_normalization_stats", False)),
            num_candidates=int(s["num_candidates"]),
            random_seed=int(s["random_seed"]),
            step_multiplier=float(s["step_multiplier"]),
            step_q=float(s["step_q"]),
            gap_q=float(s["gap_q"]),
            bounds_buffer_deg=float(s["bounds_buffer_deg"]),
        )
        out = load_model(study_id, "moveformer", cov_dim=int(getattr(mf_test, "cov_dim", 0)))
        if out is not None:
            models["moveformer"], metas["moveformer"] = out

    # didn't ask for this
    # deepssf_loader = DataLoader(deepssf_test, batch_size=int(dcfg.get("batch_size", 32)), shuffle=False, 
    #                             num_workers=8, #  int(issf_cfg.get("num_workers", 0) or 0), 
    #                             collate_fn=collate_deepssf)
    # if "deepssf" in models:
    #     r = eval_deepssf(models["deepssf"], deepssf_loader, DEVICE, topk=TOPK)
    #     summary.append({"study_id": study_id, "kind": "deepssf", **r})

    print("Selecting step starts...")
    df = pd.read_csv(study.trajectories_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    steps = select_step_starts(df[df[SET_COL] == "test"].copy(), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    if len(steps) == 0:
        continue
    n = int(len(steps))
    
    # didn't ask for this
    # if SURFACE_STEPS_PER_STUDY is None:
    #     idxs = list(range(n))
    # else:
    #     k = min(int(SURFACE_STEPS_PER_STUDY), n)
    #     idxs = sorted(int(x) for x in rng.choice(n, size=k, replace=False).tolist())
    idxs = list(range(n))

    print("Computing movement features...")
    df_tr = compute_movement_features(df[df[SET_COL] == "train"].copy(), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    _sampler, sampler_state, _ = mb_points.build_train_sampler(df_tr, parametric=True, num_candidates=int(ISSF_SURFACE_NUM_CANDIDATES), random_seed=int(s["random_seed"]), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    io = study.layer_io(str(layers_config))
    layer_slices = {layer: plan.layer_slice(layer) for layer in plan.layers_in_order}
    px = float(getattr(chips_test, "pixel_size_m"))

    print("Sampling and computing metrics...")
    # iSSF: threaded extract + score + sample (no IPC)
    if ("issf_linear" in models) or ("issf_mlp" in models):
        import threading
        from concurrent.futures import ThreadPoolExecutor, as_completed

        tls = threading.local()

        def get_io():
            if not hasattr(tls, "io"):
                tls.io = study.layer_io(str(layers_config))
            return tls.io

        reqs = [
            mb_points._StepRequest(
                point_id_start=int(r.point_id_start),
                timestamp=pd.Timestamp(r.timestamp),
                start_lon=float(r.start_lon),
                start_lat=float(r.start_lat),
                start_bearing=float(r.start_bearing),
                end_lon=float(r.end_lon),
                end_lat=float(r.end_lat),
            )
            for r in steps.iloc[idxs].itertuples(index=False)
        ]
        items = list(zip(idxs, reqs))

        def work(item):
            step_idx, req = item
            env_np, valid_np, sl_np, ta_np, _pid, tsn = mb_points._extract_one_step(
                req,
                io=get_io(),
                plan=plan,
                layer_slices=layer_slices,
                sampler_state=sampler_state,
                bounds_buffer_deg=float(s["bounds_buffer_deg"]),
            )
            env = torch.from_numpy(env_np).float()[:, :-1]
            valid = torch.from_numpy(valid_np).bool()[:, :-1]
            sl = np.asarray(sl_np[:-1], dtype=np.float32)
            ta = np.asarray(ta_np[:-1], dtype=np.float32)
            x_step = issf_step_x(env, valid, torch.from_numpy(sl), torch.from_numpy(ta), int(tsn), mv, mean, std, issf_cfg).unsqueeze(0).to(DEVICE)
            headings = float(req.start_bearing) + ta
            dx_all = sl * np.sin(headings)
            dy_all = sl * np.cos(headings)
            out = []
            with torch.no_grad():
                for kind in ("issf_linear", "issf_mlp"):
                    if kind not in models:
                        continue
                    logits = models[kind](x_step).squeeze(0).detach().cpu()
                    prob = torch.softmax(logits, dim=0)
                    g = torch.Generator(device="cpu")
                    g.manual_seed(seed(SURFACE_SEED, study_id, step_idx, kind))
                    samp = torch.multinomial(prob, int(ISSF_SURFACE_NUM_SAMPLES), replacement=True, generator=g).numpy()
                    out.extend(
                        {"study_id": study_id, "step_idx": int(step_idx), "kind": kind, "dx_m": float(dx), "dy_m": float(dy)}
                        for dx, dy in zip(dx_all[samp], dy_all[samp])
                    )
            return out

        with ThreadPoolExecutor(max_workers=8) as ex:
            futs = [ex.submit(work, item) for item in items]
            for fut in tqdm(as_completed(futs), total=len(futs), desc="issf surface"):
                surface.extend(fut.result())


    # deepSSF: batched inference over all idxs
    if "deepssf" in models:
        import os

        nw = min(8, os.cpu_count() or 1, int(dcfg.get("num_workers", 0) or 0))
        loader = DataLoader(
            Subset(deepssf_test, idxs),
            batch_size=int(dcfg.get("batch_size", 32)),
            shuffle=False,
            num_workers=nw,
            collate_fn=collate_deepssf,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=nw > 0,
        )
        pos = 0
        with torch.no_grad():
            for x1, x2, x3, _y in tqdm(loader, desc="deepssf surface"):
                step_batch = idxs[pos : pos + int(x1.shape[0])]
                pos += int(x1.shape[0])
                out = models["deepssf"]((x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE)))
                scores = out[..., 0] + out[..., 1]
                B, H, W = scores.shape
                prob = torch.softmax(scores.reshape(int(B), int(H) * int(W)).detach().cpu(), dim=1)
                for bi, step_idx in enumerate(step_batch):
                    g = torch.Generator(device="cpu")
                    g.manual_seed(seed(SURFACE_SEED, study_id, step_idx, "deepssf"))
                    pix = torch.multinomial(prob[int(bi)], int(DEEPSSF_SURFACE_NUM_SAMPLES), replacement=True, generator=g).numpy()
                    rr, cc = (pix // int(W)).astype(int), (pix % int(W)).astype(int)
                    x_coords_m, y_coords_m = centered_grid_coordinates_m((H, W), px)
                    dx = x_coords_m[cc]
                    dy = y_coords_m[rr]
                    surface.extend(
                        {"study_id": study_id, "step_idx": int(step_idx), "kind": "deepssf", "dx_m": float(ddx), "dy_m": float(ddy)}
                        for ddx, ddy in zip(dx, dy)
                    )

    # MoveFormer: batched inference over segments
    if ("moveformer" in models) and (mf_test is not None):
        import os

        perm = mf_test._cand_perm.numpy()
        sl = mf_test.step_length_m.numpy().astype(np.float32, copy=False)[:, perm]
        ta = mf_test.turn_angle_rad.numpy().astype(np.float32, copy=False)[:, perm]
        sb = np.asarray(mf_test._starts.start_bearing, dtype=np.float32)
        heading = sb[:, None] + ta
        dx_cand = sl * np.sin(heading)
        dy_cand = sl * np.cos(heading)

        P = int(dx_cand.shape[1])
        logits_steps = np.full((n, P), np.nan, dtype=np.float32)
        mask_steps = np.zeros((n,), dtype=np.bool_)

        class _MoveFormerIdx:
            def __init__(self, base):
                self.base = base
            def __len__(self):
                return len(self.base)
            def __getitem__(self, i):
                return int(i), self.base[int(i)]

        def _collate_moveformer_idx(batch):
            seg_ids = [b[0] for b in batch]
            feats = [b[1] for b in batch]
            return seg_ids, collate_moveformer(feats)

        mfcfg = dict((cfg.get("datasets", {}) or {}).get("moveformer", {}) or {})
        nw = min(8, os.cpu_count() or 1, int(mfcfg.get("num_workers", 0) or 0))
        loader = DataLoader(
            _MoveFormerIdx(mf_test),
            batch_size=int(mfcfg.get("batch_size", 1)),
            shuffle=False,
            num_workers=nw,
            collate_fn=_collate_moveformer_idx,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=nw > 0,
        )

        with torch.no_grad():
            for seg_ids, batch in tqdm(loader, desc="moveformer logits"):
                features = {k: v.to(DEVICE) for k, v in batch.items()}
                logits, _loss, _metrics = models["moveformer"](features)
                mask = features.get("loss_mask", features["mask"]).to(torch.bool).detach().cpu().numpy()
                logits = logits.detach().cpu().numpy()
                for bi, seg_id in enumerate(seg_ids):
                    start, end = mf_test._segments[int(seg_id)]
                    L = int(end - start)
                    logits_steps[start:end] = logits[int(bi), :L]
                    mask_steps[start:end] = mask[int(bi), :L]

        for step_idx in tqdm(idxs, desc="moveformer surface"):
            if not bool(mask_steps[int(step_idx)]):
                continue
            prob = torch.softmax(torch.from_numpy(logits_steps[int(step_idx)]), dim=0)
            if int(prob.numel()) > 1:
                prob = prob.clone()
                prob[0] = 0.0
                sprob = prob.sum()
                if float(sprob) <= 0:
                    prob[1:] = 1.0
                    prob = prob / prob.sum()
                else:
                    prob = prob / sprob
            g = torch.Generator(device="cpu")
            g.manual_seed(seed(SURFACE_SEED, study_id, int(step_idx), "moveformer"))
            samp = torch.multinomial(prob, int(ISSF_SURFACE_NUM_SAMPLES), replacement=True, generator=g).numpy()
            surface.extend(
                {"study_id": study_id, "step_idx": int(step_idx), "kind": "moveformer", "dx_m": float(dx), "dy_m": float(dy)}
                for dx, dy in zip(dx_cand[int(step_idx), samp], dy_cand[int(step_idx), samp])
            )



# df_sum = pd.DataFrame(summary).sort_values(["study_id", "kind"]).reset_index(drop=True)
df_surf = pd.DataFrame(surface)
p_sum, p_surf = out_cmp / f"{study_id}_summary.csv", out_cmp / f"{study_id}_surface_samples.csv"
# df_sum.to_csv(p_sum, index=False)
df_surf.to_csv(p_surf, index=False)
print("Wrote:", p_sum)
print("Wrote:", p_surf)
# display(df_sum.head(20))
# display(df_surf.head(20))


/data/vision/beery/scratch/justin/envs/movebench/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cfg: /data/vision/beery/scratch/justin/movebench/configs/benchmarks/all.yaml
layers_config: /data/vision/beery/scratch/justin/movebench/configs/layers_all.yaml
output_dir: /data/vision/beery/scratch/justin/movebench/results/benchmarks
checkpoint_root: /data/vision/beery/scratch/justin/movebench/results/benchmarks/checkpoints
n_studies: 1 ['tsavo_lions']
Computing for tsavo_lions
Selecting step starts...
Computing movement features...
Sampling and computing metrics...


steps: 100%|██████████| 815/815 [25:35<00:00,  1.88s/it]


Wrote: /data/vision/beery/scratch/justin/movebench/results/benchmarks/compare_outputs/tsavo_lions_summary.csv
Wrote: /data/vision/beery/scratch/justin/movebench/results/benchmarks/compare_outputs/tsavo_lions_surface_samples.csv


In [3]:
import numpy as np, pandas as pd
from movebench.geometry import local_xy_m_from_lonlat
from movebench import Study
from movebench.trajectories import SET_COL, TIME_COL, select_step_starts

surf = df_surf.copy()  # has: study_id, step_idx, kind, dx_m, dy_m
sampler = cfg["datasets"]["issf"]["sampler"]

truth = []
for sid in surf["study_id"].astype(str).unique():
    st = Study(sid)
    df = pd.read_csv(st.trajectories_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    steps = select_step_starts(
        df[df[SET_COL] == "test"].copy(),
        step_multiplier=float(sampler["step_multiplier"]),
        step_q=float(sampler["step_q"]),
        gap_q=float(sampler["gap_q"]),
    )
    dx_true_m, dy_true_m = local_xy_m_from_lonlat(
        steps["start_lon"].to_numpy(),
        steps["start_lat"].to_numpy(),
        steps["end_lon"].to_numpy(),
        steps["end_lat"].to_numpy(),
    )
    truth.append(pd.DataFrame({
        "study_id": sid,
        "step_idx": np.arange(len(steps), dtype=int),
        "dx_true_m": np.asarray(dx_true_m),
        "dy_true_m": np.asarray(dy_true_m),
    }))

truth = pd.concat(truth, ignore_index=True)
df_surf_l2 = surf.merge(truth, on=["study_id", "step_idx"], how="left")
df_surf_l2["l2_m"] = np.hypot(df_surf_l2["dx_m"] - df_surf_l2["dx_true_m"],
                              df_surf_l2["dy_m"] - df_surf_l2["dy_true_m"])
df_surf_l2.head()


,study_id,step_idx,kind,dx_m,dy_m,dx_true_m,dy_true_m,l2_m
0,tsavo_lions,0,issf_linear,118.377838,-1108.077637,1657.291183,-1828.551273,1699.216450
1,tsavo_lions,0,issf_linear,-1372.453979,-5196.822266,1657.291183,-1828.551273,4530.408947
2,tsavo_lions,0,issf_linear,-1434.474243,-17.072327,1657.291183,-1828.551273,3583.360075
3,tsavo_lions,0,issf_linear,571.535889,-57.899361,1657.291183,-1828.551273,2077.034606
4,tsavo_lions,0,issf_linear,-28.475298,-0.489673,1657.291183,-1828.551273,2486.688127


In [4]:
# df_surf_l2[df_surf_l2.kind=='issf_linear'].sort_values('l2_m',ascending=False)
df_surf_l2[df_surf_l2.kind=='issf_linear'].l2_m.mean(), df_surf_l2[df_surf_l2.kind=='issf_mlp'].l2_m.mean(), df_surf_l2[df_surf_l2.kind=='deepssf'].l2_m.mean()

(np.float64(5697.634148129661),
 np.float64(5408.163268075148),
 np.float64(5522.940087041177))

In [5]:
import numpy as np, pandas as pd
from movebench import Study
from movebench.trajectories import SET_COL, TIME_COL, POINT_ID_COL, select_step_starts, compute_movement_features
import movebench.features.points as mb_points

truth = df_surf_l2[["study_id", "step_idx", "dx_true_m", "dy_true_m"]].drop_duplicates()
s = cfg["datasets"]["issf"]["sampler"]

rows = []
rw_frames = []
for sid in truth["study_id"].astype(str).unique():
    st = Study(sid)
    df = pd.read_csv(st.trajectories_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    df_test = df[df[SET_COL] == "test"].copy()
    df_train = df[df[SET_COL] == "train"].copy()

    d_train = compute_movement_features(
        df_train.copy(),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
    )
    _sampler, rw_state, _ = mb_points.build_train_sampler(
        d_train,
        parametric=True,
        num_candidates=int(ISSF_SURFACE_NUM_SAMPLES),
        random_seed=int(s["random_seed"]),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
    )

    steps = select_step_starts(
        df_test.copy(),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
    )
    d = compute_movement_features(
        df_test.copy(),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
    )
    sl_prev = d.set_index(POINT_ID_COL)["step_length_m"]  # prev-step length into each point (m)

    for i in sorted(truth.loc[truth.study_id == sid, "step_idx"].astype(int).unique()):
        rows.append({"study_id": sid, "step_idx": int(i), "kind": "baseline_nomove", "dx_m": 0.0, "dy_m": 0.0})
        r = steps.iloc[int(i)]
        sl = float(sl_prev.get(int(r.point_id_start), np.nan))
        b = float(r.start_bearing)
        rows.append({"study_id": sid, "step_idx": int(i), "kind": "baseline_constvel", "dx_m": sl * np.sin(b), "dy_m": sl * np.cos(b)})
        lens, turns = mb_points._sample_pseudo_steps(rw_state, int(r.point_id_start))
        headings = b + turns
        rw_frames.append(pd.DataFrame({"study_id": sid, "step_idx": int(i), "kind": "baseline_randomwalk", "dx_m": lens * np.sin(headings), "dy_m": lens * np.cos(headings)}))

baselines = pd.concat([pd.DataFrame(rows)] + rw_frames, ignore_index=True).merge(truth, on=["study_id", "step_idx"], how="left")
baselines["l2_m"] = np.hypot(baselines["dx_m"] - baselines["dx_true_m"], baselines["dy_m"] - baselines["dy_true_m"])

df_surf_l2_plus = pd.concat([df_surf_l2, baselines], ignore_index=True)
display(df_surf_l2_plus.groupby("kind")["l2_m"].agg(["count", "mean", "median", "std"]).sort_values("mean"))


,count,mean,median,std
kind,,,,
baseline_nomove,815,3920.212761,2543.883478,4264.873047
issf_mlp,815000,5408.163268,4227.999193,4798.796326
baseline_constvel,815,5427.913118,4545.244924,4705.769110
deepssf,407500,5522.940087,4230.314223,4873.019124
issf_linear,815000,5697.634148,4300.963042,5384.197755


In [7]:
import numpy as np
import pandas as pd
import torch

# Energy score (L2): ES = E||X - y|| - 0.5 * E||X - X'||
def _energy_l2_group(g: pd.DataFrame) -> pd.Series:
    g = g.dropna(subset=["dx_m", "dy_m", "dx_true_m", "dy_true_m"])
    n = int(len(g))
    if n == 0:
        return pd.Series({"n": 0, "energy_l2": np.nan})

    y = torch.tensor([g["dx_true_m"].iloc[0], g["dy_true_m"].iloc[0]], dtype=torch.float32)
    x = torch.tensor(g[["dx_m", "dy_m"]].to_numpy(), dtype=torch.float32)

    term1 = torch.linalg.norm(x - y, dim=1).mean()
    term2 = torch.tensor(0.0) if n == 1 else torch.cdist(x, x).mean()
    return pd.Series({"n": n, "energy_l2": float(term1 - 0.5 * term2)})

df_energy = (
    df_surf_l2_plus
    .groupby(["study_id", "step_idx", "kind"], sort=False)
    .apply(_energy_l2_group)
    .reset_index()
)

display(df_energy.groupby("kind")["energy_l2"].agg(["count", "mean", "median", "std"]).sort_values("mean"))
# df_energy.head()


,count,mean,median,std
kind,,,,
issf_mlp,815,3112.953949,1857.848633,3333.443882
deepssf,815,3153.671089,1930.816895,3295.214068
issf_linear,815,3212.342588,1954.216064,3451.074373
baseline_nomove,815,3920.212761,2543.883545,4264.873058
baseline_constvel,815,5427.913116,4545.244629,4705.769087
